# Chapter 13 — Persistence & Streaming in LangGraph

Hands-On LangChain. Code targets **LangGraph 1.x**.

When you build agents for more than a single turn, two concepts become essential: **persistence** and
**streaming**. They are the bedrock of robust, production-ready applications that handle long-running
tasks, maintain context, and give a transparent user experience.

**Persistence** is the ability to save the state of your agent at any point, so you can pause and resume
conversations — a form of memory, and the basis for human-in-the-loop. **Streaming** gives a real-time
view into the agent's operation: the intermediate steps (which tools are being called) and the final
answer as it's generated, token by token.

In this chapter we take a basic LangGraph agent and add both capabilities.

Companion notebook for Chapter 13. Configure Ollama and pull the chat model set in `MODEL` in `.env`.

> Two modernizations from the source article: the search tool is **DuckDuckGo** (no API key) rather than
> Tavily, and persistence uses LangGraph's built-in **`InMemorySaver`** — which does exactly what the
> article's in-memory `SqliteSaver(":memory:")` did, with no extra package, and works for both sync and
> async streaming.

## 13.1 Setting up the agent

This is the same research-assistant agent from the previous chapters, equipped with a search tool. The
state is just a list of messages we append to over time.

In [1]:
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from model_provider import get_chat_model
from langchain_community.tools import DuckDuckGoSearchResults

load_dotenv(find_dotenv(usecwd=True))

tool = DuckDuckGoSearchResults(num_results=2, output_format="list")

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

## 13.2 Adding persistence with checkpointers

To add persistence, LangGraph introduces the **checkpointer**: it automatically saves a snapshot of the
graph's state after every step (after each node runs), so the agent can pick up right where it left off.

We'll use **`InMemorySaver`**, a temporary in-memory checkpointer perfect for testing. (For production
you'd point a checkpointer at SQLite, Postgres, or Redis instead.) The only change to our `Agent` class
is a `checkpointer` parameter passed to `graph.compile()`.

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.add_edge(START, "llm")
        self.graph = graph.compile(checkpointer=checkpointer)   # <-- persistence
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_model(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        return {"messages": [self.model.invoke(messages)]}

    def exists_action(self, state: AgentState):
        return len(state["messages"][-1].tool_calls) > 0

    def take_action(self, state: AgentState):
        results = []
        for t in state["messages"][-1].tool_calls:
            print(f"Calling: {t['name']} {t['args']}")
            result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        print("Back to the model!")
        return {"messages": results}

In [3]:
prompt = """You are a smart research assistant. Use the search engine to look up information. You are allowed to make multiple calls (either together or in sequence). Only look up information when you are sure of what you want. If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = get_chat_model(temperature=0)
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

## 13.3 Streaming intermediate steps

To manage conversational history we introduce a **thread** — a unique identifier for a conversation, so
the checkpointer can keep multiple conversations independent. We define it with a `configurable` key
holding a `thread_id`.

Instead of `invoke`, we use **`stream`**, which yields the state updates at each step. We print the last
message of each step with `.pretty_print()` for readability.

In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}

for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        v["messages"][-1].pretty_print()

We see the agent's reasoning unfold: an AI message that decides to call the search tool, the tool's
result, then the final AI answer.

Now let's use the persistence. We ask a **follow-up** on the **same `thread_id`**, so the agent has the
previous messages as context.

In [ ]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        v["messages"][-1].pretty_print()

The agent infers we're still talking about the weather and looks up Los Angeles. One more, still on thread 1:

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        v["messages"][-1].pretty_print()

It uses the whole conversation to compare the two cities. To prove the `thread_id` is what makes
this work, let's ask the same question on a **new thread (`"2"`)** — with no shared history.

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        v["messages"][-1].pretty_print()

The model is now confused and asks for clarification — a new thread has no access to the prior conversation. This is exactly how checkpointers and threads enable stateful, multi-turn interactions.

## 13.4 Streaming tokens in real time

Besides streaming the intermediate *steps*, we can stream the final answer **token by token**, like
ChatGPT does. This uses the asynchronous **`astream_events`** method. We look for events of kind
`on_chat_model_stream` and print each non-empty chunk (empty chunks signal a tool call rather than text).

`InMemorySaver` supports async, so we can reuse the same agent.

In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}

async for event in abot.graph.astream_events({"messages": messages}, thread, version="v2"):
    if event["event"] == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            print(content, end="|")

Each `|` marks a streamed chunk, so you can watch the answer build in real time — a far more
interactive experience than waiting for the whole response.

By combining persistence and streaming, you elevate a LangGraph agent from a one-shot tool into a
stateful application. Persistence (via checkpointers) gives the memory needed for coherent, long-running
conversations and is the key enabler of the **human-in-the-loop** workflows we explore next. Streaming —
of both steps and tokens — offers transparency and a dynamic user experience.